# Run this cell first

In [ ]:
# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

try:
  from testsrc import test_main
except (ModuleNotFoundError, ImportError):
  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=MTH2100/numerical_analysis/interpolation"
  from testsrc import test_main

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except (AssertionError, ImportError):
      pass

# Interpolation: Reconstructing Functions from Data

## Why this matters

In modelling, we often work with data rather than closed-form functions:
- Experimental measurements at discrete time points
- Tabulated values (e.g of material properties)
- Coarse simulation output that needs to be evaluated at intermediate points

**Interpolation** finds a function that passes exactly through all given data points.
Choosing the right interpolation method is important — the wrong choice can introduce
large spurious oscillations.

**Learning objectives**
- Implement piecewise linear interpolation
- Implement Lagrange polynomial interpolation and observe Runge's phenomenon
- Understand cubic splines and when to use them
- Apply interpolation to a real-world dataset

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;">

**📹 VIDEO: Interpolation: From Linear to Splines**

*What this video covers:* The interpolation problem vs curve fitting; piecewise linear interpolation; Lagrange polynomial interpolation; the danger of high-degree polynomial interpolation (Runge's phenomenon); cubic splines and their advantages; choosing an interpolation method.

*(Embed the video here using an `<iframe>` or `IPython.display.Video`)*
</div>

## 1. The Interpolation Problem

Given $n+1$ data points $(x_0, y_0), (x_1, y_1), \ldots, (x_n, y_n)$ with distinct
$x_i$, find a function $p(x)$ such that:

$$p(x_i) = y_i \quad \text{for all } i = 0, 1, \ldots, n$$

I.E find a curve that passes through all points exactly. This is called _interpolation_. It is an alternative to regression or fitting which
don't aim to agree exactly with the data, but rather to minimise the error.

Key question: *what class of functions should $p$ belong to?*
- Polynomials: simple to work with (especially, e.g. for finding derivatives)
- Piecewise linear: extremely simple conceptually (straight lines between points)
- Cubic splines: smooth, local, and well-behaved. 

To see the kinds of issues that can arise, let's work through an example. You can read about the methods for constructing the different interpolants elsewhere (e.g [Burden and Faires](https://qub.primo.exlibrisgroup.com/discovery/fulldisplay?docid=alma991004422139708046&context=L&vid=44QSUB_INST:QUB&lang=en&search_scope=QUB_Campus&adaptor=Local%20Search%20Engine&tab=QUB_Campus&query=any,contains,burden%20faires)). Here we'll just use existing libraries to build the functions we need. 

---

## The target 'function'

Let's imagine that we are measuring something: we point our telescope at the night sky and measure the position of a distant planet relative to its star as it orbits. From our side-on perspective it looks like the planet is just oscillating from side to side. Of course, we can't measure a continuous function- so we measure discrete values of this position at some rate. E.G once every couple of minutes. What we end up with is a list of numbers:

| time (s)| position on screen (mm)| 
|---|---|
|0.0 | 0.0 |
| 120 | 6.5 |
| 240 | 9.6 | 
| 360 | 8.2 |
| ... | ... |

Thankfully, the observatory is keeping track of the numbers for us so, we can simply load the data into our python notebook, and begin to try and understand the pattern in the numbers.

In [ ]:
import numpy as np

x_data, y_data = np.loadtxt('https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/MTH2100/numerical_analysis/interpolation/interp_data.txt')



The first thing we should do is plot the data to see if we can spot what's going on

In [ ]:

import matplotlib.pyplot as plt

plt.plot(x_data, y_data, 'ro', label='measurements')
plt.xlabel('time (hours)')
plt.ylabel('on screen position (cm)')

This looks kind of weird, but at least we can see where the planet is! The trouble is, we only 'know' where the planet is at these precise instants in time. If we want to know where it is at some point in time 'in between' these data points, we're out of luck. The process of finding out what's happening in-between the points (or 'poles') is called _interpolation_. 

---

The simplest thing we can do with a set of $N+1$ points is find the $N^{th}$-degree polynomial that passes through those points. You've been doing this for years already without realising it. If you're given 2 points, you can find a unique straight line that passes through them. If you have three points, there is precisely one quadratic function that passes through those points. Finding this polynomial can be done in various ways. The one I learned as a student was called Lagrange Interpolation, which is encoded in the formulas

$$p(x) = \sum_{k=0}^n y_k \ell_k(x), \qquad
\ell_k(x) = \prod_{\substack{j=0\\j\neq k}}^n \frac{x - x_j}{x_k - x_j}$$

where the 'basis polynomial' $\ell_k(x)$ equals 1 at $x_k$ and 0 at all other data points. When you multiply all the terms together (that's what the $\prod$ symbol means) you get a polynomial of degree $n$ which is guaranteed to match the target data at the poles. Finding these basis polynomials is really boring, by the way. Another way you can find the polynomial is by solving a big matrix equation: 

$$
\begin{pmatrix}
1 & x_0 & x_0^2 & \cdots & x_0^n \\
1 & x_1 & x_1^2 & \cdots & x_1^n \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_n & x_n^2 & \cdots & x_n^n
\end{pmatrix}
\begin{pmatrix} a_0 \\ a_1 \\ \vdots \\ a_n \end{pmatrix}
=
\begin{pmatrix} y_0 \\ y_1 \\ \vdots \\ y_n \end{pmatrix}
$$

to find the polynomial coefficients $a_0, a_1, \ldots$. This is called the [Vandermonde matrix](https://en.wikipedia.org/wiki/Vandermonde_matrix) by the way. Solving that's even more boring. But at least a computer can do it quite quickly. Thankfully, there are built-in functions for doing polynomial interpolation with python.

In [ ]:
from scipy.interpolate import BarycentricInterpolator

lagrange_polynomial = BarycentricInterpolator(x_data, y_data)

# check the function can calculate a value for x not in x_data
print(lagrange_polynomial(0.7))

# Calculate the value of the polynomial at each x-value in x_data, and compare it to the actual y-value in y_data:
print(lagrange_polynomial(x_data) - y_data)

Happy days! We have a function that equals our data at the measurement points, and gives us values in between. Job done, right? Let's plot the function and see.

In [ ]:
x_fine = np.linspace(x_data[0], x_data[-1], 1000) # 'fine grid' of values
y_lagrange = lagrange_polynomial(x_fine)

plt.plot(x_data, y_data, 'ro', label='measurements')
plt.plot(x_fine, y_lagrange, 'b-', label='lagrange polynomial')
plt.legend()
plt.axis([0, 2, -2, 2])
plt.xlabel('time (hours)')
plt.ylabel('on screen position (cm)')

Ah. Not so good. The problem is that we have a really high order (in this case 59th order) polynomial. It turns out, if you raise a number to its 59th power, it gets pretty big. This would be fine if our data behaved like a polynomial, but it doesn't seem to. Just because the function passes through all the right points, doesn't mean it's behaving the right way in between. 

--- 


Let's try a different approach. Instead of trying to get one 'global' function which matches the function at all the points, lets find functions which agree 'locally'. This is what is called a piecewise function: one which has a different behaviour on different regions. 

The simplest interpolation is simply to 'draw' straight lines between all the points. This is actually what matplotlib does by default if you plot a list of numbers: try changing the plot command above so that instead of plotting red circles `ro`, the data is plotted with a red line `r-`. 

But just plotting the straight lines still doesn't tell us what the values are in between measurements, doesn't allow us to do things like compute derivatives to solve differential equations: for that we need a function. The technical name for the function is a 'piecewise linear spline' function, i.e. at any point, the function behaves as a straight line, and the lines all connect to each other. If we were to write down the functions, it would look something like

$$ y = \begin{cases} m_1 x + c_1 \text{ for } 0 \leq x \leq 0.1 \\
                     m_2 x + c_2 \text{ for } 0.1 \leq x \leq 0.2 \\
                     \ldots \end{cases} $$

(By the way, a 'spline' was (is) a flexible piece of wood that you clamp between two or more fixed points to form a curve. It was a used traditionally for things like making the hulls of boats: you made a rough frame of 'points' and then bent the splines between those clamped points, giving you a smooth curve made up of several splines, and used that to draw the shape you wanted.)

Anyway, we don't need to faff about with actually calculating what all the gradients ($m$) and y-intercepts ($c$) are in the formulas, nor do we need to do this with bits of wood. Instead, we can use a built-in function. While we're at it we'll do something slightly more sophisticated as well. Instead of having straight lines connecting the points, we'll use cubic functions. The reason for this is that with straight lines, where two points connect, the function is continuous, but not smooth. With piecewise cubic functions we can enforce that the function and its first couple of derivatives are continuous, i.e. smooth. (The reason cubics are the standard, by the way, is because it turns out that's what the pieces of wood do naturally if you let them bend between the clamps: a cubic curve minimises the elastic energy stored in the wood).
 

In [ ]:
from scipy.interpolate import make_interp_spline

linear_spline = make_interp_spline(x_data, y_data, k=1) # k=1 is the order of the spline, i.e. 'order 1 = linear'
cubic_spline = make_interp_spline(x_data, y_data, k=3) 

print(linear_spline(0.7))
print(cubic_spline(0.7))
print(linear_spline(x_data) - y_data)

Brilliant: again, we have functions which agree with the measured data, and which will give us the in-between values. You can see though that they give slightly different values for the value of the function at $x=0.7$, which is to be expected- they're different functions, let's plot them to see what's happening.

In [ ]:
y_linear = linear_spline(x_fine)
y_cubic = cubic_spline(x_fine)

plt.plot(x_data, y_data, 'ro', label='measurements')

plt.plot(x_fine, y_linear, 'b-', label='Linear Spline')
plt.plot(x_fine, y_cubic, 'g-', label='Cubic Spline')

plt.xlabel('time (hours)')
plt.ylabel('on screen position (cm)')
plt.axis([0, 2, -2, 2])
plt.legend()


Now we're getting somewhere. The cubic probably looks more visually pleasing to you, because it behaves more like a smooth function the way we would expect things (like planets) to behave in nature. However, you always have to remember that the only thing we're sure of is the measurements; the linear spline _could_ be more like the true underlying function, but the only way we would know is to make more measurements, or make predictions based on the different interpolants, and see if they bear out.

That caveat made, once we have an interpolating function like a cubic spline, we can use it for other things- like filling in gaps in our data, or calculating rates of change, or integrating (we'll come on to that next) to (e.g.) solve a differential equation. 

---

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

There are a few gotchas with interpolating polynomials. One we have already seen- a high-order interpolating polynomial can go a little crazy in between the interpolating points. This is actually known as Runge's phenomenon. The Runge function is given by $$ f(x) = \frac{1}{25 - x^2}.$$

1. Define a function called `runge` which takes one input argument `x` and returns the value of the Runge function at that value of $x$. 
2. define `x5` and `x11` as linearly spaced arrays between $-20$ and $+20$ with 5 and 11 values respectively. Use your `runge` function to compute `y5` and `y11`: the values of the Runge function at the values of $x$ contained in the `x5` and `x11` arrays.  
3. Use the `BarycentricInterpolator` to compute the order 4 and order 10 polynomial interpolants which match the Runge function at the $x$ values in `x5` and `x11` respectively. Call these polynomial functions `poly5` and `poly11` respectively. 
4. Define an array `x_fine` which contains 500 linearly spaced values between -20 and +20. Use it with your `runge`, `poly5` and `poly11` functions to plot the behaviour of the three functions.

**Question:** Can you explain why the interpolating polynomials do such a bad job of representing the function? Where is the representation particularly bad, and why? Can you suggest a strategy for fixing that problem? 

<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f7d6e3; border-left:4px solid #f32121; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

The fact is, the Runge function doesn't behave like a polnomial. As $x$ gets larger, the Runge function gets smaller, but for polynomials the opposite is true. This means that the interpolating polynomials perform particularly badly towards the ends of the interval, i.e. as $x$ approaches $-20$ and $+20$. One strategy for making this better would be to use an alternative interpolating approach, like we did above with (e.g) cubic splines. Another, which might be non-obvious is to use unequally spaced points for the interpolation. We used equally spaced points, but the interpolant seems to have a harder time fitting the function towards the ends. We can help it out by giving it more data points where things are hard, and fewer where it's easy. There's a proper way of doing this with [chebyshev nodes](https://en.wikipedia.org/wiki/Chebyshev_nodes) but a cheap way to do it is just manually select 11 points with an unequal spacing, like, 

```python
xbias = np.array([-20, -19, -16, -12, -7, 0 , 7, 12, 16, 19, 20])
polyb = BarycentricInterpolator(xbias, runge(xbias))
plt.plot(x_fine, polyb(x_fine))
```
If you plot that alongside the other interpolating polynomials, it looks like this 

![](./runge.png)

I.E with the same number of interpolating points, but choosing slightly smarter, we can get much closer agreement with the target function.
</details>
</div>

</div>

In [ ]:
# your code goes here





# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()

In [ ]:
runtest(['test_runge', 'test_xarrays', 'test_yarrays', 'test_plot'])

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

A second gotcha has to do with the data itself. Depending on how (or when, or where) the measurements are taken, we can end up with an impossible task. To demonstrate that 

1. Define a function `oscillator` which takes one input variable `x` and returns the value of the function $f(x) = \sin(20x)$
2. Set up two arrays, one called `x20` with 20 equally spaced points between $0$ and $2\pi$ and one called `x21` with 21 equally spaced points over the same range
3. Use `oscillator` to plot the value of the function at the points in `x20` and `x21`
   
**Question** without even doing an interpolation for these curves, can you see an issue with trying to fit these data? 
<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f7d6e3; border-left:4px solid #f32121; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

The data with 20 equally spaced points looks straightforward to fit: it looks like a sine wave. And we know that the underlying function is a sine wave, so what's the problem? The problem is that the underlying function is $\sin(20x)$ and what is represented by these points looks like $\sin(x)$. The one with 21 equally spaced points is even worse- it looks like the function is zero everywhere! If we try to fit these with interpolating polynomials, we will certainly get something that fits these data points, but it won't look anything like the underlying function. This is not a problem with the interpolation, but a problem with the data. We have been unfortunate that the 'sampling rate' of our 'measurement' happens to match up 'perfectly' with the frequency of the underlying signal. To see what the function is supposed to look like, you can plot it with a much finer grid of x-values.

Have you ever wondered why, on TV, car wheels often appear to rotate more slowly than they should, or even to remain stationary, or to rotate backwards? This is why! It's called aliasing, and it happens when the sampling rate of a measurement (in this case, the number of frames per second captured by the video camera) is not sufficiently high to capture the dynamics. Between one frame and the next, the wheel might have completed almost one full rotation, but that means that to our eye, it looks to have moved slightly backwards. 


</div>

In [ ]:
# your code goes here





# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()

In [ ]:
runtest(['test_alias'])

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise
When we make a measurement, we rarely get things exactly right. In other words, real data has noise. An exact interpolating spline can [overfit](https://en.wikipedia.org/wiki/Overfitting). `scipy.interpolate.UnivariateSpline` supports a smoothing parameter $s$ that controls the trade-off between smoothness and fitting the data exactly. With the data loaded for you in the cell below, use this `UnivariateSpline` function as follows,

```python
cubic_with_smoothing = UnivariateSpline(x_noisy, y_noisy, s=0.1)
y1 = cubic_with_smoothing(x)
```

(I.E the same way we used the `make_interp_spline` function earlier, but with the extra `s` parameter) to produce a plot with several different values for the smoothing: `s=0`, `s=1` and `s=0.1`. The plot should contain four data sets, the raw data, and the three interpolating cubic splines with smoothing.

**Question:** Describe how the $s$ parameter affects the resulting spline fit. What are the tradeoffs for using various levels of smoothing in a real context?
<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f7d6e3; border-left:4px solid #f32121; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

$s=0$ should give you the same behaviour as you get with `make_interp_spline` -- i.e. no smoothing -- the fitted function should look exactly like the data. With a large value of $s$, all of the noise is filtered out, you get something that looks perfectly smooth. For an intermediate value, $s=0.1$ you still get some fluctuations, but any rapid oscillations are filtered out. 

The tradeoffs are entirely dependent on your data and your application. If you know that what you should be seeing is smooth (e.g. the motion of objects in space where there aren't any frictional forces), you can probably safely smooth the data. Similarly, if you know that the noise is unimportant for your application (e.g. you want to predict long-term trends in the stock-market) then you can also ignore the noise. But if you don't know what the underlying function is, or you don't know whether a particular feature is 'signal' or 'noise' the best thing to do is to play with the models you build- see what happens if you smooth the data or don't smooth it. It might turn out that a detail has no impact on the final result, in which case it can be safely ignored. Or it may be super important, in which case you might start to look for the reasons why. 

</div>
</details>

</div>

</div>

In [ ]:
from scipy.interpolate import UnivariateSpline

x_noisy, y_noisy = np.loadtxt('https://raw.githubusercontent.com/autofeedback-exercises/exercises/main/MTH2100/numerical_analysis/interpolation/noisy_data.txt')
plt.plot(x_noisy, y_noisy, 'ro')
x_fine = np.linspace(0, 2*np.pi, 100)


# Calculate the three different splines and plot them using x_fine as the x values




# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()


In [ ]:
runtest(['test_noisy'])